In [21]:
import pandas as pd


df = pd.DataFrame({
    "record_id": [101, 102, 103, 104, 105, 106, 107, 108],

    "device_id": [
        10001, 10002, 10003, 10004,
        10005, 10006, 10007, 10008
    ],

    "site": [
        "R34", "R35", "R36", "R34",
        "R35", "R36", "R34", "R35"
    ],

    "created_at": [
        "2026-08-13 08:15:00",
        "2026-08-13 09:30:00",
        "2026-08-13 10:45:00",
        "invalid_time",
        "2026-08-13 12:10:00",
        "2026-08-14 07:20:00",
        "2026-08-14 08:40:00",
        "2026-08-14 09:50:00"
    ],

    "temperature": [
        "26.5",
        "27.1",
        "ERROR",
        "28.3",
        "--",
        "25.8",
        "26.9",
        "27.5"
    ],

    "visibility": [
        "1200",
        "850",
        "600",
        "1500",
        "950",
        "700",
        "1100",
        "500"
    ]
})

df

,record_id,device_id,site,created_at,temperature,visibility
0,101,10001,R34,2026-08-13 08:15:00,26.5,1200
1,102,10002,R35,2026-08-13 09:30:00,27.1,850
2,103,10003,R36,2026-08-13 10:45:00,ERROR,600
3,104,10004,R34,invalid_time,28.3,1500
4,105,10005,R35,2026-08-13 12:10:00,--,950
5,106,10006,R36,2026-08-14 07:20:00,25.8,700
6,107,10007,R34,2026-08-14 08:40:00,26.9,1100
7,108,10008,R35,2026-08-14 09:50:00,27.5,500


In [22]:
df_cleaned = df.copy()

## 任务 1：检查当前数据类型

查看所有字段当前的数据类型。

思考：

哪些字段的数据类型明显不合理？

In [23]:
df_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   record_id    8 non-null      int64
 1   device_id    8 non-null      int64
 2   site         8 non-null      str  
 3   created_at   8 non-null      str  
 4   temperature  8 non-null      str  
 5   visibility   8 non-null      str  
dtypes: int64(2), str(4)
memory usage: 743.0 bytes


## 任务 2：处理时间字段

将：

`created_at`

转换成真正的日期时间类型。

要求：

无法解析的时间不要让程序报错，而是转换为缺失时间。

In [24]:
df_cleaned['created_at'] = pd.to_datetime(df_cleaned['created_at'],errors='coerce')
df_cleaned['created_at']

0   2026-08-13 08:15:00
1   2026-08-13 09:30:00
2   2026-08-13 10:45:00
3                   NaT
4   2026-08-13 12:10:00
5   2026-08-14 07:20:00
6   2026-08-14 08:40:00
7   2026-08-14 09:50:00
Name: created_at, dtype: datetime64[us]

## 任务 3：处理温度字段

`temperature`

理论上应该是数值，但其中存在：

- 正常数字字符串
- `"ERROR"`
- `"--"`

将其转换成数值类型。

无法转换的值设为缺失值。

In [25]:
df_cleaned['temperature'] = pd.to_numeric(df_cleaned['temperature'],errors='coerce')
df_cleaned['temperature']

0    26.5
1    27.1
2     NaN
3    28.3
4     NaN
5    25.8
6    26.9
7    27.5
Name: temperature, dtype: float64

## 任务 4：处理能见度字段

将：

`visibility`

转换成数值类型。

转换后检查该字段的数据类型。

In [26]:
df_cleaned['visibility'] = pd.to_numeric(df_cleaned['visibility'],errors='coerce')
df_cleaned['visibility'].dtype

dtype('int64')

## 任务 5：处理设备编号

`device_id`

虽然内容看起来都是数字，但它本质上是：

> 设备标识符，而不是可以进行数学运算的数值。

将其转换成字符串类型。

In [27]:
df_cleaned['device_id'] = df_cleaned['device_id'].astype('string')

## 任务 6：生成时间分析字段

完成时间类型转换后，从：

`created_at`

中生成两个新字段：

- `date`：记录所属日期
- `hour`：记录发生的小时

In [28]:
df_cleaned['date'] = df_cleaned['created_at'].dt.date
df_cleaned['hour'] = df_cleaned['created_at'].dt.hour

df_cleaned

,record_id,device_id,site,created_at,temperature,visibility,date,hour
0,101,10001,R34,2026-08-13 08:15:00,26.5,1200,2026-08-13,8.0
1,102,10002,R35,2026-08-13 09:30:00,27.1,850,2026-08-13,9.0
2,103,10003,R36,2026-08-13 10:45:00,NaN,600,2026-08-13,10.0
3,104,10004,R34,NaT,28.3,1500,NaT,NaN
4,105,10005,R35,2026-08-13 12:10:00,NaN,950,2026-08-13,12.0
5,106,10006,R36,2026-08-14 07:20:00,25.8,700,2026-08-14,7.0
6,107,10007,R34,2026-08-14 08:40:00,26.9,1100,2026-08-14,8.0
7,108,10008,R35,2026-08-14 09:50:00,27.5,500,2026-08-14,9.0


## 任务 7：数据质量检查

完成所有转换以后：

1. 再次查看整个 DataFrame 的数据类型；
2. 找出由于类型转换失败而产生缺失值的所有记录。

In [29]:
df_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   record_id    8 non-null      int64         
 1   device_id    8 non-null      string        
 2   site         8 non-null      str           
 3   created_at   7 non-null      datetime64[us]
 4   temperature  6 non-null      float64       
 5   visibility   8 non-null      int64         
 6   date         7 non-null      object        
 7   hour         7 non-null      float64       
dtypes: datetime64[us](1), float64(2), int64(2), object(1), str(1), string(1)
memory usage: 708.0+ bytes


In [33]:
check_cols = [
    "created_at",
    "temperature",
    "visibility"
]

invalid_records = df_cleaned[
    df_cleaned[check_cols].isna().any(axis=1)
]

invalid_records

,record_id,device_id,site,created_at,temperature,visibility,date,hour
2,103,10003,R36,2026-08-13 10:45:00,NaN,600,2026-08-13,10.0
3,104,10004,R34,NaT,28.3,1500,NaT,NaN
4,105,10005,R35,2026-08-13 12:10:00,NaN,950,2026-08-13,12.0
